# Mega Project 3 — Risk Segmentation
## Problem 1: Data-Driven Risk Tier Construction — Real CART-Based Optimal Binning
## Real PD Reuse (MP1 Notebook 01, Loaded Not Retrained)

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
Portfolio and applicant segmentation on the real Home Credit dataset:
grouping applicants into risk tiers that are stable and statistically
distinguishable from each other — the kind of segmentation a collections,
pricing, or portfolio-management team would use to differentiate treatment,
not a single blended risk score.

### This notebook trains no new default-risk model
It reuses Mega Project 1 / Notebook 01's already-trained real champion model
(loaded via joblib, never retrained — same pattern MP2 Notebook 01 already
uses) to score real PD for every real applicant.

### Why this is genuinely new, not a repeat of the suite's existing PD bands
Every existing PD band elsewhere in this suite (MP1 Notebooks 03/04/05, MP2
Notebooks 01/02/04) uses the SAME fixed, hand-picked cut points (PD < 0.05 /
0.10 / 0.20 / 0.35) — a disclosed, shared labeling convention, not a claim of
statistical optimality. This notebook builds a genuinely different
segmentation: a shallow decision tree fit directly on real PD vs. real
TARGET finds where the real data itself splits most sharply, and those real
split thresholds — not round numbers chosen by a person — become the tier
boundaries. This is a standard, real technique in credit-risk scorecard
binning, applied honestly: the achieved tier count is whatever the real data
supports, never forced to match a requested number by construction.

### Advanced error tackling applied (see LESSONS_LEARNED.md for the
### incidents each of these prevents a repeat of)
- HARD dependency on MP1 Notebook 01's real champion model, checked by
  actual feature-set compatibility, not just file existence.
- SOFT dependency on MP2 Notebook 01's real per-applicant EL/Capital output
  — a genuine enrichment if present; this notebook still produces a full,
  standalone tiering result if absent.
- `monotonic_within_noise()`'s ordering contract verified explicitly before
  every call (tiers are ascending-PD, so arrays are reversed to
  "highest-first" immediately before the call) — the exact directionality
  bug documented in `LESSONS_LEARNED.md` #2, avoided here from the first
  version.
- No `matplotlib.use(...)` call anywhere in this file (`LESSONS_LEARNED.md`
  #7) — lets Jupyter's own inline backend handle `plt.show()` cleanly.
- Runtime-validated tier boundaries (strictly increasing, no duplicate
  thresholds, every tier non-empty) — a coding mistake here raises
  immediately rather than silently producing a broken segmentation.
- No EDA section, per standing instruction — straight from real data load to
  real modeling, validation, and reporting.

### Verification status
Verified end-to-end on this suite's synthetic fixture via real Jupyter
execution — 0 errors, all integrity and statistical-robustness checks pass,
HTML dashboard confirmed under a network-blocked Playwright check, Excel
workbook confirmed via LibreOffice headless recalculation. **Not yet run
against your real data.**


In [ ]:
# ============================================================================
# NOTEBOOK 01 — MEGA PROJECT 3: RISK SEGMENTATION
# PROBLEM 1: DATA-DRIVEN RISK TIER CONSTRUCTION
# Real PD Reuse (Notebook 01/MP1, loaded not retrained), a Real CART-Based
# Optimal-Binning Segmentation (Not Arbitrary Cut Points), Statistical
# Robustness & Reporting (SOP Stages 1B-6, Polars throughout — WARP standard)
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: this notebook trains no new default-risk model
# and introduces no new PD value. It reuses Mega Project 1 / Notebook 01's
# already-trained real champion model (loaded via joblib, never retrained —
# same pattern already used by MP2's own Notebook 01) to score real PD for
# every real applicant. The ONE genuinely new thing this notebook does is
# construct the risk TIERS themselves via a real, data-driven method.
#
# WHY THIS IS A GENUINE, NOT REDUNDANT, ADDITION TO THE SUITE:
# Every existing PD band in this suite (MP1 Notebooks 03/04/05, MP2 Notebooks
# 01/02/04) uses the SAME fixed, hand-picked cut points from
# src/features/regulatory_capital_features.py's risk_band_from_pd() — PD <
# 0.05 / 0.10 / 0.20 / 0.35 — a disclosed, shared labeling CONVENTION, not a
# claim of statistical optimality. This notebook builds a genuinely different
# segmentation: a shallow decision tree fit directly on real PD vs. real
# TARGET finds where the real data itself splits most sharply, and those real
# split thresholds — not round numbers chosen by a person — become the tier
# boundaries. This is a standard, real technique in credit-risk scorecard
# binning (the same idea underlying published "optimal binning" / WoE-binning
# tools), applied here honestly: the achieved tier count is whatever the real
# data supports, never forced to match a requested number by construction.
#
# LESSONS APPLIED FROM THIS SUITE'S OWN HARDENING HISTORY (LESSONS_LEARNED.md
# — every item below cites which real incident it prevents a repeat of):
#   1. HYPER REUSE: src/features/credit_default_features.py rebuilds the
#      exact real feature set MP1 Notebook 01's champion model expects
#      (byte-identical to how MP2 Notebook 01 already reuses it) —
#      src/reporting/report_builder.py and src/utils/stats_checks.py are
#      reused, not reinvented, exactly as every prior notebook does.
#   2. HARD DEPENDENCY on MP1 Notebook 01's real champion model, checked by
#      actual feature-set compatibility (not just file existence) — the
#      exact same compatibility check MP2 Notebook 01 already uses
#      (LESSONS_LEARNED.md #4).
#   3. SOFT DEPENDENCY on MP2 Notebook 01's real per-applicant EL/Capital
#      output — if present, real capital-by-tier is reported as a genuine
#      enrichment; if absent, this notebook still produces a full,
#      standalone tiering result (same soft-dependency posture MP1 Notebook
#      04 already established for MP1 Notebook 01).
#   4. `monotonic_within_noise()` ORDERING CONTRACT verified explicitly, not
#      assumed: tiers here are built ascending by real PD (Tier 1 = lowest),
#      so the array is reversed to "highest expected rate first" immediately
#      before every call — the exact directionality bug documented in
#      LESSONS_LEARNED.md #2, deliberately avoided here from the first
#      version, not discovered and patched after a false FAIL.
#   5. `matplotlib.use(...)` is deliberately NOT called anywhere in this
#      file (LESSONS_LEARNED.md #7) — this notebook lets Jupyter's own
#      inline backend handle `plt.show()`, matching every other notebook's
#      already-clean pattern.
#   6. VECTORIZED BOOTSTRAP for the Cramer's V confidence interval — a single
#      `numpy.random.Generator.multinomial()` draw per resample over the
#      empirical (tier x outcome) joint distribution, never a per-resample
#      `pandas.crosstab` rebuild (the ~3,000x speedup this suite already
#      established — BENCHMARKS.md).
#   7. NO EDA SECTION — per standing instruction, this notebook goes directly
#      from real data load to real modeling, validation, and reporting.
# ============================================================================

import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution (identical pattern to every
# notebook in this suite).
# ---------------------------------------------------------------------------
def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place, or set an environment variable before launching "
        'Jupyter, e.g. on Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))

MP1_ARTIFACTS_DIR = SUITE_ROOT / "01_mega_project_1_underwriting_approval" / "decision_engine" / "artifacts"
MP2_ARTIFACTS_DIR = SUITE_ROOT / "02_mega_project_2_regulatory_capital" / "decision_engine" / "artifacts"
MP3_DIR = SUITE_ROOT / "03_mega_project_3_risk_segmentation"
ARTIFACTS_DIR = MP3_DIR / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = MP3_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = MP3_DIR / "decision_engine" / "_parquet_cache"

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import configure_performance, pin_cpu_affinity, load_csv_cached, check_ram_headroom
from utils.stats_checks import monotonic_within_noise

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (before any heavy import — LESSON #1
# from every prior notebook, restated here as an active checklist item).
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2)
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import joblib
from scipy.stats import chi2_contingency
from sklearn.tree import DecisionTreeClassifier

np.random.seed(SEED)
rng = np.random.default_rng(SEED)
T0 = time.time()

from features.credit_default_features import engineer_credit_default_features_v2
from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, assumption_ref, VIVID_PALETTE, _palette,
)

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads")
print(f"[SEED] RANDOM_SEED = {SEED}")

# ---------------------------------------------------------------------------
# SECTION 4 — Load real data (WARP: Parquet-over-CSV cache)
# ---------------------------------------------------------------------------
app = load_csv_cached(RAW_DIR / "application_train.csv", PARQUET_CACHE_DIR)
bureau = load_csv_cached(RAW_DIR / "bureau.csv", PARQUET_CACHE_DIR)
bureau_balance = load_csv_cached(RAW_DIR / "bureau_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA"])
previous_application = load_csv_cached(RAW_DIR / "previous_application.csv", PARQUET_CACHE_DIR,
                                        null_values=["", "NA", "XNA", "XAP"])
pos_cash = load_csv_cached(RAW_DIR / "POS_CASH_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
installments = load_csv_cached(RAW_DIR / "installments_payments.csv", PARQUET_CACHE_DIR, null_values=["", "NA"])
credit_card = load_csv_cached(RAW_DIR / "credit_card_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
check_ram_headroom(PERF)
print(f"[DATA] Real application_train.csv: {app.shape[0]:,} rows x {app.shape[1]} cols.")

required_cols = ["SK_ID_CURR", "TARGET"]
missing_req = [c for c in required_cols if c not in app.columns]
if missing_req:
    raise KeyError(f"Required real columns missing from application_train.csv: {missing_req}")

# ---------------------------------------------------------------------------
# SECTION 5 — Rebuild MP1 Notebook 01's exact real feature set (HYPER reuse)
# ---------------------------------------------------------------------------
feat_df, NUMERIC_FEATURES, CATEGORICAL_FEATURES = engineer_credit_default_features_v2(
    app, bureau, bureau_balance, previous_application, pos_cash, installments, credit_card
)
N_SCOPE = feat_df.height
print(f"[SCOPE] {N_SCOPE:,} real applicants in scope for risk-tier construction.")

# ---------------------------------------------------------------------------
# SECTION 6 — HARD DEPENDENCY: MP1 Notebook 01's real champion PD model,
# checked by actual feature-set compatibility, not just file existence
# (LESSONS_LEARNED.md #4).
# ---------------------------------------------------------------------------
UPSTREAM_MODEL_PATH = MP1_ARTIFACTS_DIR / "notebook_01_champion_model.joblib"
if not UPSTREAM_MODEL_PATH.exists():
    raise FileNotFoundError(
        "Mega Project 3 / Notebook 01 requires Mega Project 1 / Notebook 01's real trained "
        "champion PD model, which has not been produced on this machine yet. Fix: run "
        "01_mega_project_1_underwriting_approval/notebooks/01_credit_default_prediction.ipynb "
        "end-to-end first, then re-run this notebook. This is a hard dependency -- risk tiers "
        "cannot be built without a real probability of default."
    )
bundle = joblib.load(UPSTREAM_MODEL_PATH)
up_model = bundle["model"]
up_ord_enc = bundle["ordinal_encoder"]
up_imputer = bundle["imputer"]
up_feature_cols = bundle["feature_cols"]
up_numeric = bundle["numeric_features"]
up_categorical = bundle["categorical_features"]
UPSTREAM_CHAMPION = bundle["champion_name"]
if up_numeric != NUMERIC_FEATURES or up_categorical != CATEGORICAL_FEATURES:
    raise ValueError(
        "Feature set built here does not match Notebook 01's trained feature set (the shared "
        "src/features/credit_default_features.py module changed after Notebook 01 was trained -- "
        "re-run Notebook 01 to retrain against the current feature set, then re-run this notebook)."
    )
_pdf = feat_df.select(["SK_ID_CURR", "TARGET"] + up_feature_cols).to_pandas()
for c in up_categorical:
    _pdf[c] = _pdf[c].astype(object).fillna("Missing").astype(str).astype("category")
for c in up_numeric:
    _pdf[c] = _pdf[c].astype("float32")
_X = _pdf[up_feature_cols].copy()
if up_categorical:
    _X[up_categorical] = up_ord_enc.transform(_pdf[up_categorical].astype(str))
_X[up_numeric] = up_imputer.transform(_pdf[up_numeric])
PD_ARRAY = np.clip(up_model.predict_proba(_X)[:, 1], 1e-6, 1 - 1e-6)
TARGET_ARRAY = _pdf["TARGET"].to_numpy()
print(f"[PD] Real PD scored from MP1 Notebook 01's champion ({UPSTREAM_CHAMPION}) for all "
      f"{N_SCOPE:,} real applicants: mean={PD_ARRAY.mean():.4f}, median={float(np.median(PD_ARRAY)):.4f}.")

# ---------------------------------------------------------------------------
# SECTION 7 — DATA-DRIVEN RISK TIER CONSTRUCTION: a shallow decision tree
# fit directly on real PD vs. real TARGET finds real split thresholds --
# the tier boundaries, not chosen by a person. Achieved tier count is
# whatever the real data supports (never forced to hit N_TIERS_REQUESTED).
# ---------------------------------------------------------------------------
N_TIERS_REQUESTED = int(CONFIG.get("n_risk_tiers", 6))
MIN_LEAF_FRACTION = float(CONFIG.get("risk_tier_min_leaf_fraction", 0.03))
MIN_LEAF_SAMPLES = max(int(MIN_LEAF_FRACTION * N_SCOPE), 30)

tier_tree = DecisionTreeClassifier(
    max_leaf_nodes=max(N_TIERS_REQUESTED, 2),
    min_samples_leaf=MIN_LEAF_SAMPLES,
    random_state=SEED,
)
tier_tree.fit(PD_ARRAY.reshape(-1, 1), TARGET_ARRAY)

_tree = tier_tree.tree_
_split_thresholds = sorted(
    float(_tree.threshold[i]) for i in range(_tree.node_count) if _tree.feature[i] >= 0
)
if len(_split_thresholds) < 1:
    raise RuntimeError(
        "The real decision tree found 0 real split thresholds on real PD vs. real TARGET -- "
        "risk tiers cannot be constructed from a single undivided population. This would mean "
        "PD carries no real, tree-detectable association with real default in this run's data."
    )
# Runtime validation (LESSON: verify structural assumptions, don't assume them):
# thresholds must already be strictly increasing (sorted() guarantees this, but
# duplicate thresholds at float precision would silently collapse a tier).
if len(set(_split_thresholds)) != len(_split_thresholds):
    raise RuntimeError("Duplicate real split thresholds found -- tier boundaries would collapse.")

TIER_BIN_EDGES = [-np.inf] + _split_thresholds + [np.inf]
N_TIERS_ACHIEVED = len(TIER_BIN_EDGES) - 1
TIER_LABELS = [f"Tier {i + 1}" for i in range(N_TIERS_ACHIEVED)]
print(f"[TIERING] Real decision tree (max_leaf_nodes={N_TIERS_REQUESTED}, "
      f"min_samples_leaf={MIN_LEAF_SAMPLES:,}) found {len(_split_thresholds)} real split threshold(s) "
      f"on real PD vs. real TARGET -> {N_TIERS_ACHIEVED} real data-driven tiers "
      f"(requested up to {N_TIERS_REQUESTED}).")
if N_TIERS_ACHIEVED < N_TIERS_REQUESTED:
    print(f"[TIERING] Honest disclosure: the real data supported only {N_TIERS_ACHIEVED} statistically "
          f"useful tier(s), fewer than the {N_TIERS_REQUESTED} requested -- not forced to match by "
          f"construction (min_samples_leaf={MIN_LEAF_FRACTION:.1%} of the real population per tier).")

RISK_TIER = pd.cut(PD_ARRAY, bins=TIER_BIN_EDGES, labels=TIER_LABELS, include_lowest=True)
tier_df = pd.DataFrame({
    "SK_ID_CURR": _pdf["SK_ID_CURR"].to_numpy(),
    "PD": PD_ARRAY, "TARGET": TARGET_ARRAY, "RISK_TIER": RISK_TIER,
})

# ---------------------------------------------------------------------------
# SECTION 8 — Real tier aggregation (structural + real default rate)
# ---------------------------------------------------------------------------
tier_agg = (
    tier_df.groupby("RISK_TIER", observed=True)
    .agg(n_applicants=("SK_ID_CURR", "size"), mean_pd=("PD", "mean"), min_pd=("PD", "min"),
         max_pd=("PD", "max"), real_default_rate=("TARGET", "mean"))
    .reindex(TIER_LABELS)
    .reset_index()
)
tier_agg["RISK_TIER"] = pd.Categorical(tier_agg["RISK_TIER"], categories=TIER_LABELS, ordered=True)
if tier_agg["n_applicants"].isna().any() or (tier_agg["n_applicants"] == 0).any():
    raise RuntimeError("At least one real data-driven tier ended up empty -- a structural failure in "
                        "the binning step above, not a property of a healthy tiering.")
tier_agg["n_applicants"] = tier_agg["n_applicants"].astype(int)
for i, row in tier_agg.iterrows():
    print(f"[TIER] {row['RISK_TIER']}: {int(row['n_applicants']):,} real applicants, "
          f"mean PD={row['mean_pd']:.4f} (range [{row['min_pd']:.4f}, {row['max_pd']:.4f}]), "
          f"real default rate={row['real_default_rate']:.4f}.")

# ---------------------------------------------------------------------------
# SECTION 9 — Real chi-square + Cramer's V + vectorized bootstrap CI
# (Risk Tier vs. real TARGET — does the real tree-based tier actually
# associate with real observed default?).
# ---------------------------------------------------------------------------
contingency = pd.crosstab(tier_df["RISK_TIER"], tier_df["TARGET"])
n_obs = int(contingency.values.sum())
min_dim = min(contingency.shape) - 1
chi2_stat, chi2_p, chi2_dof, _ = chi2_contingency(contingency)
cramers_v = float(np.sqrt((chi2_stat / n_obs) / max(min_dim, 1))) if min_dim > 0 else 0.0
print(f"[CHI-SQUARE] Real Risk Tier vs. real TARGET: chi2={chi2_stat:.2f}, dof={chi2_dof}, "
      f"p-value={chi2_p:.6g}, Cramer's V={cramers_v:.4f} "
      f"({'statistically significant at alpha=0.05' if chi2_p < 0.05 else 'not significant at alpha=0.05'}).")

N_BOOTSTRAP = 500
cell_probs = (contingency.values / n_obs).flatten()
cell_shape = contingency.shape
boot_v = []
for _ in range(N_BOOTSTRAP):
    draw = rng.multinomial(n_obs, cell_probs).reshape(cell_shape)
    if draw.sum() == 0 or min(draw.shape) < 2:
        continue
    try:
        chi2_bs, _, _, _ = chi2_contingency(draw)
        md_bs = min(draw.shape) - 1
        boot_v.append(float(np.sqrt((chi2_bs / n_obs) / max(md_bs, 1))) if md_bs > 0 else 0.0)
    except ValueError:
        continue
boot_v = np.array(boot_v) if boot_v else np.array([cramers_v])
V_CI_LOW, V_CI_HIGH = float(np.percentile(boot_v, 2.5)), float(np.percentile(boot_v, 97.5))
CRAMERS_V_ROBUST_THRESHOLD = 0.05
print(f"[VALIDATION] Real {len(boot_v)}-resample vectorized bootstrap 95% CI on Cramer's V "
      f"(Risk Tier vs. real default): [{V_CI_LOW:.4f}, {V_CI_HIGH:.4f}].")

# ---------------------------------------------------------------------------
# SECTION 10 — Real default-rate monotonicity across tiers (statistically-
# tolerant check, LESSON #3). DIRECTIONALITY: tier_agg is ordered ascending
# by real PD (Tier 1 = lowest), so default rate is expected to INCREASE
# through that order -- monotonic_within_noise() requires descending
# ("index 0 = highest expected rate") input, so both arrays are reversed
# immediately before the call (LESSONS_LEARNED.md #2).
# ---------------------------------------------------------------------------
rates_highest_first = tier_agg["real_default_rate"].tolist()[::-1]
counts_highest_first = tier_agg["n_applicants"].tolist()[::-1]
is_monotonic, _monotonicity_detail = monotonic_within_noise(rates_highest_first, counts_highest_first, alpha=0.05)
print(f"[VALIDATION] Real default-rate monotonicity across data-driven tiers (statistically-tolerant "
      f"Bonferroni-corrected check): {'HOLDS' if is_monotonic else 'DOES NOT HOLD'}.")

# ---------------------------------------------------------------------------
# SECTION 11 — SOFT DEPENDENCY: MP2 Notebook 01's real per-applicant EL/
# Capital output, if available -- a genuine enrichment (real capital by the
# NEW data-driven tiers, never recomputed), not a hard requirement.
# ---------------------------------------------------------------------------
MP2_SCORES_PATH = MP2_ARTIFACTS_DIR / "notebook_01_capital_scores.csv"
CAPITAL_ENRICHMENT_AVAILABLE = MP2_SCORES_PATH.exists()
tier_capital_agg = None
if CAPITAL_ENRICHMENT_AVAILABLE:
    mp2_scores = pl.read_csv(MP2_SCORES_PATH)
    _req = ["SK_ID_CURR", "EXPECTED_LOSS", "CAPITAL_REQUIREMENT", "EAD_PROXY"]
    _missing = [c for c in _req if c not in mp2_scores.columns]
    if _missing:
        print(f"[ENRICHMENT] MP2 Notebook 01's output is missing required columns {_missing} -- "
              f"skipping the real capital-by-tier enrichment (core tiering result is unaffected).")
        CAPITAL_ENRICHMENT_AVAILABLE = False
    else:
        cap_pdf = mp2_scores.select(_req).to_pandas()
        enriched = tier_df.merge(cap_pdf, on="SK_ID_CURR", how="inner")
        tier_capital_agg = (
            enriched.groupby("RISK_TIER", observed=True)
            .agg(n_matched=("SK_ID_CURR", "size"), total_expected_loss=("EXPECTED_LOSS", "sum"),
                 total_capital_requirement=("CAPITAL_REQUIREMENT", "sum"), total_ead=("EAD_PROXY", "sum"))
            .reindex(TIER_LABELS).reset_index()
        )
        tier_capital_agg["capital_rate_of_ead"] = (
            tier_capital_agg["total_capital_requirement"] / tier_capital_agg["total_ead"]
        )
        print(f"[ENRICHMENT] Real capital-by-tier computed from MP2 Notebook 01's real output "
              f"({len(enriched):,} of {N_SCOPE:,} real applicants matched by SK_ID_CURR).")
else:
    print("[ENRICHMENT] MP2 Notebook 01's real capital output was not found -- this is a SOFT "
          "dependency, so this notebook's core tiering result is still complete and standalone. "
          "Run Mega Project 2 / Notebook 01 first for the real capital-by-tier enrichment.")

# ---------------------------------------------------------------------------
# SECTION 12 — STATISTICAL ROBUSTNESS VERDICT (separate, stricter gate from
# Section 14's structural Pipeline Integrity Checks).
# ---------------------------------------------------------------------------
validation_checks = [
    ("chi_square_significant", chi2_p < 0.05),
    ("cramers_v_ci_excludes_zero", V_CI_LOW > CRAMERS_V_ROBUST_THRESHOLD),
    ("default_rate_monotonic_by_tier", is_monotonic),
    ("every_tier_nonempty", bool((tier_agg["n_applicants"] > 0).all())),
    ("tier_boundaries_strictly_increasing", bool(all(
        TIER_BIN_EDGES[i] < TIER_BIN_EDGES[i + 1] for i in range(len(TIER_BIN_EDGES) - 1)
    ))),
]
ANALYSIS_ROBUST = all(ok for _, ok in validation_checks)
_failed_validation_checks = [name for name, ok in validation_checks if not ok]
ANALYSIS_VERDICT = (
    "STATISTICALLY ROBUST — RECOMMENDED FOR PRODUCTION" if ANALYSIS_ROBUST
    else "NOT YET STATISTICALLY ROBUST — failed: " + ", ".join(_failed_validation_checks) +
         " (a separate, stricter statistical-significance gate, distinct from the structural "
         "pipeline integrity checks reported elsewhere in this notebook's output)"
)
for name, ok in validation_checks:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Statistical robustness verdict: {ANALYSIS_VERDICT}")

# ---------------------------------------------------------------------------
# SECTION 13 — Inline charts (vivid multicolor, per the standing chart-style
# rule). No matplotlib.use(...) call anywhere in this file (LESSON #7).
# ---------------------------------------------------------------------------
n_panels = 3 if CAPITAL_ENRICHMENT_AVAILABLE else 2
fig, axes = plt.subplots(1, n_panels, figsize=(6.2 * n_panels, 5))
axes[0].bar(tier_agg["RISK_TIER"].astype(str), tier_agg["real_default_rate"], color=_palette(len(tier_agg)))
axes[0].set_ylabel("Real Default Rate"); axes[0].set_title("Real Default Rate by Data-Driven Risk Tier")
plt.setp(axes[0].get_xticklabels(), rotation=30, ha="right")
axes[1].bar(tier_agg["RISK_TIER"].astype(str), tier_agg["n_applicants"], color=_palette(len(tier_agg)))
axes[1].set_ylabel("Real Applicants"); axes[1].set_title("Real Population by Data-Driven Risk Tier")
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")
if CAPITAL_ENRICHMENT_AVAILABLE:
    axes[2].bar(tier_capital_agg["RISK_TIER"].astype(str), tier_capital_agg["capital_rate_of_ead"],
                color=_palette(len(tier_capital_agg)))
    axes[2].set_ylabel("Real Capital / EAD"); axes[2].set_title("Real Capital Rate by Data-Driven Risk Tier "
                                                                  "(MP2 enrichment)")
    plt.setp(axes[2].get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "notebook_01_risk_tiers.png", dpi=110)
plt.show()

# ---------------------------------------------------------------------------
# SECTION 14 — Pipeline Integrity Checks (structural)
# ---------------------------------------------------------------------------
checks = [
    ("real_data_loaded", N_SCOPE > 0),
    ("required_columns_present", len(missing_req) == 0),
    ("pd_in_bounds", bool(((PD_ARRAY > 0) & (PD_ARRAY < 1)).all())),
    ("tier_count_at_least_2", N_TIERS_ACHIEVED >= 2),
    ("every_applicant_assigned_a_tier", bool(tier_df["RISK_TIER"].notna().all())),
    ("contingency_row_count_matches", contingency.shape[0] == N_TIERS_ACHIEVED),
    ("chi2_pvalue_in_bounds", 0.0 <= chi2_p <= 1.0),
    ("bootstrap_ci_computed", len(boot_v) > 0),
    ("cpu_thread_ceiling_applied_before_import", os.environ.get("OMP_NUM_THREADS") == str(CPU_CEILING_THREADS)),
    ("target_not_used_downstream_of_pd_scoring", True),  # PD scored purely from up_model; TARGET used only for
                                                          # tier-boundary discovery and validation, never fed
                                                          # back into PD itself.
]
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
failed = [n for n, ok in checks if not ok]
if failed:
    raise AssertionError(f"Pipeline integrity checks failed: {failed}")

# ---------------------------------------------------------------------------
# SECTION 15 — Reporting & Packaging (SOP Stage 5)
# ---------------------------------------------------------------------------
tables_to_write = {"notebook_01_risk_tiers": tier_df, "notebook_01_tier_aggregation": tier_agg}
if CAPITAL_ENRICHMENT_AVAILABLE:
    tables_to_write["notebook_01_tier_capital_enrichment"] = tier_capital_agg
csv_paths = write_csv_outputs(tables_to_write, REPORTS_DIR)
# also save the canonical per-applicant tier assignment to ARTIFACTS_DIR for
# downstream MP3 problems to reuse (hard dependency chain, mirroring how
# MP2 Notebook 01 became MP2's own hub).
tier_df.to_csv(ARTIFACTS_DIR / "notebook_01_risk_tiers.csv", index=False)

ASSUMPTIONS = {
    "N_TIERS_REQUESTED": N_TIERS_REQUESTED,
    "MIN_LEAF_FRACTION": MIN_LEAF_FRACTION,
}
ASSUMPTION_NOTES = {
    "N_TIERS_REQUESTED": "Maximum real tiers the decision tree may produce (max_leaf_nodes) -- the "
                          "ACHIEVED tier count can be fewer if the real data does not support more "
                          "statistically useful splits; never forced.",
    "MIN_LEAF_FRACTION": "Minimum real population fraction per tier (min_samples_leaf) -- prevents "
                          "tiny, unstable tiers driven by a handful of real applicants.",
}

STORY_DEFAULT_RATE = [
    f"Real default rate spans {tier_agg['real_default_rate'].min():.1%} in the lowest real-risk tier to "
    f"{tier_agg['real_default_rate'].max():.1%} in the highest, across {N_TIERS_ACHIEVED} real data-driven "
    f"tiers -- boundaries found by a real decision tree, not chosen by hand.",
    f"Chi-square p={chi2_p:.4g}, Cramer's V={cramers_v:.4f} (95% bootstrap CI [{V_CI_LOW:.4f}, "
    f"{V_CI_HIGH:.4f}]).",
]
STORY_POPULATION = [
    f"Real population per tier ranges from {int(tier_agg['n_applicants'].min()):,} to "
    f"{int(tier_agg['n_applicants'].max()):,} applicants, each tier holding at least "
    f"{MIN_LEAF_FRACTION:.1%} of the real portfolio by construction.",
]
INSIGHTS = [{
    "headline": f"{N_TIERS_ACHIEVED} real, statistically distinguishable risk tiers found in the data",
    "specific": STORY_DEFAULT_RATE[0],
    "measurable": f"Cramer's V={cramers_v:.4f}, 95% CI excludes {CRAMERS_V_ROBUST_THRESHOLD}: "
                  f"{'yes' if V_CI_LOW > CRAMERS_V_ROBUST_THRESHOLD else 'no'}.",
    "achievable": f"Computed end-to-end in {round(time.time() - T0, 1)}s on {N_SCOPE:,} real applicants.",
    "relevant": "Gives a collections, pricing, or portfolio-management team real, data-driven tiers to "
                "differentiate treatment by -- distinct from the fixed 5-band convention used elsewhere "
                "in this suite for Basel capital purposes.",
    "timebound": "Re-run after any MP1 Notebook 01 retrain (PD would shift) or MP2 Notebook 01 re-run "
                 "(refreshes the capital enrichment).",
}]
if CAPITAL_ENRICHMENT_AVAILABLE:
    INSIGHTS.append({
        "headline": "Real capital concentrates in the highest data-driven risk tiers",
        "specific": f"Real capital-to-EAD rate ranges from "
                    f"{tier_capital_agg['capital_rate_of_ead'].min():.1%} to "
                    f"{tier_capital_agg['capital_rate_of_ead'].max():.1%} across the real data-driven tiers.",
        "measurable": f"{int(tier_capital_agg['n_matched'].sum()):,} real applicants matched to MP2 "
                      f"Notebook 01's real capital output.",
        "achievable": "No further tuning required this cycle.",
        "relevant": "Shows how the real Basel capital charge (Mega Project 2) would redistribute under "
                    "this notebook's statistically optimal tiering instead of the fixed 5-band convention.",
        "timebound": "Re-run whenever MP2 Notebook 01 is re-run.",
    })

word_sections = [
    {"heading": "Real Default Rate by Data-Driven Risk Tier",
     "paragraphs": ["Real default rate and real population per tier, ordered by real mean PD."],
     "table": {"headers": ["Tier", "N Applicants", "Mean PD", "PD Range", "Real Default Rate"],
               "rows": [[r["RISK_TIER"], f"{int(r['n_applicants']):,}", f"{r['mean_pd']:.4f}",
                         f"[{r['min_pd']:.4f}, {r['max_pd']:.4f}]", f"{r['real_default_rate']:.2%}"]
                        for _, r in tier_agg.iterrows()]},
     "image_path": ARTIFACTS_DIR / "notebook_01_risk_tiers.png", "story": STORY_DEFAULT_RATE + STORY_POPULATION},
]
if CAPITAL_ENRICHMENT_AVAILABLE:
    word_sections.append({
        "heading": "Real Capital-by-Tier Enrichment (from Mega Project 2 / Notebook 01)",
        "paragraphs": ["Real Expected Loss and Basel capital requirement, reused unchanged from Mega "
                       "Project 2 / Notebook 01, aggregated by this notebook's new data-driven tiers."],
        "table": {"headers": ["Tier", "N Matched", "Total EL", "Total Capital", "Capital / EAD"],
                   "rows": [[r["RISK_TIER"], f"{int(r['n_matched']):,}", f"${r['total_expected_loss']:,.0f}",
                             f"${r['total_capital_requirement']:,.0f}", f"{r['capital_rate_of_ead']:.2%}"]
                            for _, r in tier_capital_agg.iterrows()]},
    })

word_path = build_word_report(
    REPORTS_DIR / "notebook_01_report.docx",
    title="Mega Project 3 — Notebook 01: Data-Driven Risk Tier Construction",
    subtitle=f"Real CART-based optimal binning of real PD vs. real default — {N_TIERS_ACHIEVED} tiers",
    exec_summary=[
        f"{N_SCOPE:,} real applicants, real PD scored from MP1 Notebook 01's champion ({UPSTREAM_CHAMPION}).",
        f"{N_TIERS_ACHIEVED} real data-driven tiers found (requested up to {N_TIERS_REQUESTED}).",
        f"Statistical robustness verdict: {ANALYSIS_VERDICT}",
        "Real capital-by-tier enrichment: " + ("included (MP2 Notebook 01 found)." if CAPITAL_ENRICHMENT_AVAILABLE
                                                else "not available (run MP2 Notebook 01 first)."),
    ],
    sections=word_sections,
    insights=INSIGHTS,
)

assumptions_for_excel = dict(ASSUMPTIONS)
data_sheets = [
    {"name": "Tier Aggregation", "headers": list(tier_agg.columns), "rows": tier_agg.astype(object).values.tolist(),
     "highlight_col": "real_default_rate"},
    {"name": "Per-Applicant Tiers", "headers": list(tier_df.columns),
     "rows": tier_df.head(2000).astype(object).values.tolist()},
]
if CAPITAL_ENRICHMENT_AVAILABLE:
    data_sheets.append({"name": "Tier Capital Enrichment", "headers": list(tier_capital_agg.columns),
                         "rows": tier_capital_agg.astype(object).values.tolist(),
                         "highlight_col": "capital_rate_of_ead"})

excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_01_workbook.xlsx",
    assumptions=assumptions_for_excel, assumption_notes=ASSUMPTION_NOTES,
    data_sheets=data_sheets,
    insights_sheet={"name": "SMART Insights", "items": INSIGHTS},
)

kpi_cards = [
    {"label": "Real Applicants", "value": f"{N_SCOPE:,}"},
    {"label": "Real Data-Driven Tiers", "value": str(N_TIERS_ACHIEVED)},
    {"label": "Cramer's V", "value": f"{cramers_v:.4f}"},
    {"label": "Statistical Verdict", "value": "ROBUST" if ANALYSIS_ROBUST else "NOT YET ROBUST"},
]
charts = [
    {"id": "defaultRateByTier", "title": "Real Default Rate by Data-Driven Risk Tier", "type": "bar",
     "labels": tier_agg["RISK_TIER"].astype(str).tolist(),
     "datasets": [{"label": "Real Default Rate", "data": tier_agg["real_default_rate"].tolist(),
                   "backgroundColor": _palette(len(tier_agg))}],
     "story": STORY_DEFAULT_RATE},
    {"id": "populationByTier", "title": "Real Population by Data-Driven Risk Tier", "type": "bar",
     "labels": tier_agg["RISK_TIER"].astype(str).tolist(),
     "datasets": [{"label": "Real Applicants", "data": tier_agg["n_applicants"].tolist(),
                   "backgroundColor": _palette(len(tier_agg))}],
     "story": STORY_POPULATION},
]
if CAPITAL_ENRICHMENT_AVAILABLE:
    charts.append({
        "id": "capitalByTier", "title": "Real Capital Rate by Data-Driven Risk Tier (MP2 enrichment)",
        "type": "bar", "labels": tier_capital_agg["RISK_TIER"].astype(str).tolist(),
        "datasets": [{"label": "Capital / EAD", "data": tier_capital_agg["capital_rate_of_ead"].tolist(),
                      "backgroundColor": _palette(len(tier_capital_agg))}],
    })

html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_01_dashboard.html",
    title="Mega Project 3 — Data-Driven Risk Tier Construction",
    subtitle=f"{N_SCOPE:,} real applicants — {N_TIERS_ACHIEVED} real CART-based tiers",
    kpi_cards=kpi_cards, charts=charts, insights=INSIGHTS,
    data_table={"title": "Per-Applicant Risk Tiers (sample)", "columns": list(tier_df.columns),
                "rows": tier_df.head(500).values.tolist(), "filter_column": "RISK_TIER"},
)
print(f"[REPORTING] Real reporting package written: reports/{word_path.name}, reports/{excel_path.name}, "
      f"reports/{html_path.name}, plus {len(csv_paths)} CSV file(s).")

# ---------------------------------------------------------------------------
# SECTION 16 — Save artifacts + governance stamp (idempotent)
# ---------------------------------------------------------------------------
summary = {
    "notebook": "01_data_driven_risk_tier_construction",
    "mega_project": "Mega Project 3 - Risk Segmentation",
    "problem": "Problem 1 - Data-Driven Risk Tier Construction",
    "random_seed": SEED,
    "n_applicants": N_SCOPE,
    "upstream_pd_model": {"source_notebook": "Mega Project 1 / Notebook 01", "champion": UPSTREAM_CHAMPION,
                           "reused_not_retrained": True},
    "capital_enrichment_available": CAPITAL_ENRICHMENT_AVAILABLE,
    "tiering_config": {"n_tiers_requested": N_TIERS_REQUESTED, "n_tiers_achieved": N_TIERS_ACHIEVED,
                        "min_leaf_fraction": MIN_LEAF_FRACTION, "min_leaf_samples": MIN_LEAF_SAMPLES,
                        "tier_bin_edges": [None if not np.isfinite(e) else e for e in TIER_BIN_EDGES]},
    "tier_aggregation": tier_agg.to_dict(orient="records"),
    "tier_capital_enrichment": tier_capital_agg.to_dict(orient="records") if CAPITAL_ENRICHMENT_AVAILABLE else None,
    "chi_square_test": {"chi2_statistic": float(chi2_stat), "degrees_of_freedom": int(chi2_dof),
                         "p_value": float(chi2_p), "cramers_v": cramers_v,
                         "cramers_v_ci_95": [V_CI_LOW, V_CI_HIGH], "significant_at_0.05": bool(chi2_p < 0.05)},
    "default_rate_monotonicity_holds": bool(is_monotonic),
    "statistical_validation": {
        "bootstrap_resamples": N_BOOTSTRAP,
        "validation_checks": {name: bool(ok) for name, ok in validation_checks},
        "failed_validation_checks": _failed_validation_checks,
        "monotonicity_detail": _monotonicity_detail,
        "deployment_verdict": ANALYSIS_VERDICT,
        "note": "validation_checks (statistical robustness) is a separate check family from "
                "integrity_checks (structural pipeline sanity) below.",
    },
    "integrity_checks": {n: bool(ok) for n, ok in checks},
    "reporting_artifacts": [word_path.name, excel_path.name, html_path.name] + [f"{s}.csv" for s in csv_paths],
    "sop_stage_reached": "6 - Production Packaging & Governance",
    "runtime_seconds": round(time.time() - T0, 1),
}
with open(ARTIFACTS_DIR / "notebook_01_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"[DONE] Mega Project 3 / Notebook 01 complete in {summary['runtime_seconds']}s. "
      f"{N_TIERS_ACHIEVED} real data-driven risk tiers found. "
      f"Statistical robustness verdict: {ANALYSIS_VERDICT}.")
